# Bedside

> Processing utilities for bedside/ICU data

In [ ]:
#| default_exp bedside

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np, zarr, datetime as dt, warnings, pandas as pd, torch

from pathlib import Path
import datetime as dt

from torch.utils.data import Dataset
from physiojepa.data_preprocessing import calculate_samples_mp, interpolate_nan_clip, calculate_samples_forecast_mp, open_zarr_group, close_zarr_group, zarr_record_name
from physiojepa.signal import butterworth, iqr_normalization, resample_waveform
from scipy.ndimage import median_filter

CLIP_INTERPOLATE_RANGES = {'ABP': {"phys_range":None, "percentiles":None}, 
                           'II': {"phys_range":None, "percentiles":None}, 
                           'V': {"phys_range":None, "percentiles":None}, 
                           'I': {"phys_range":None, "percentiles":None}, 
                           'III': {"phys_range":None, "percentiles":None},
                           'AVR': {"phys_range":None, "percentiles":None},
                           'AVF': {"phys_range":None, "percentiles":None},
                           'PLETH': {"phys_range":None, "percentiles":None},
                           'RESP': {"phys_range":None, "percentiles":None},
                           'SPO2': {"phys_range":None, "percentiles":None},
                           }

## Dataloaders

In [ ]:
#| export
class ForecastingDataset(Dataset):
    """
    PyTorch Dataset for forecasting tasks using physiological waveform data stored in zarr format.
    This dataset class handles loading, preprocessing, and preparing samples for forecasting tasks.
    """
    def __init__(self, 
                 channels, # channels to use
                 forecast_window_sec, # forecast window (within), suggest 5, 10, 15 minutes
                 outcome_df, # pandas dataframe containing outcomes for zarr files
                 outcome_df_outcome_col, # outcome column in the y file path
                 file_col='file_path', # column indicating zarr file path
                 y_date_column='date', # column indicating date of sample collection
                 outcome_df_seconds_since_column='Time Stamp (seconds)', # column indicating how many seconds since beginning of waveform
                 outcome_df_duration_column='event_length', # column indicating duration of outcome in seconds
                 sample_df=None, # dataframe indicating which indices within each zarr file includes a sample
                 sample_seq_len_sec=None, # if no sample_df, generate sequences of this length in seconds as one sample
                 frequency=125, # frequency of underlying data
                 butterworth_filters=None, # dictionary of low pass, high pass, and bandpass dictionary to perform on channels
                 median_filter_kernel_size=None, # size of median filter to perform on channels
                 clip_interpolations=None, # dictionary of channels:{'phys_range':..., 'percentiles':...} for filtering and interpolation of filtered values
                 constant_nan_tolerance=0.5, # tolerance for nan values in the data - 0 means no nan allowed, 1 means 100% of nans allowed
                 require_all_channels=False, # indicator to require all channels to be present in the sample, if False, will return samples with any of the channels and 0s for the missing channels
                 infer_forecast_windows=True, # indicator to require all forecast windows to be present in the sample, if False, will return samples with any of the forecast windows and NAs for the missing forecast windows
                 normalize_signals=True, # indicator to normalize signals to 0 mean and unit variance
                 sample_frequency_key='sampling_frequency',
                 sample_generation_workers=None # processes used when sample_df must be generated
                 ):
        self.channels = channels
        self.forecast_window_sec = [forecast_window_sec] if isinstance(forecast_window_sec, int) else forecast_window_sec
        self.sample_seq_len_sec = sample_seq_len_sec
        self.frequency = frequency
        self.clip_interpolations = clip_interpolations
        self.butterworth_filters = butterworth_filters
        self.median_filter_kernel_size = median_filter_kernel_size
        self.normalize_signals = normalize_signals
        REASON_CODES = {'1': 'Missing channels', 
                        '2': 'No channels', 
                        '3': 'Invalid indices',
                        '4': 'Missing forecast windows', 
                        '5': 'Missing due to nan or constant signal',
                        '6': 'No valid samples'
                        }

       
        if sample_df is None:
            print(f"Calculating samples with {sample_seq_len_sec} sec length using the outcome df with a forecast window of {forecast_window_sec} sec")
            self.sample_df, _, removal_reasons = calculate_samples_forecast_mp(outcome_df=outcome_df, file_col=file_col, outcome_val_col=outcome_df_outcome_col, outcome_time_col=outcome_df_seconds_since_column, outcome_duration_col=outcome_df_duration_column, forecast_window_sec=forecast_window_sec, channels=channels, frequency=frequency, sample_seq_len_sec=sample_seq_len_sec, constant_nan_tolerance=constant_nan_tolerance, require_all_channels=require_all_channels, infer_forecast_windows=infer_forecast_windows, sample_frequency_key=sample_frequency_key, n_processes=sample_generation_workers)
            self.removal_reasons = {REASON_CODES[k]:v for k,v in removal_reasons.items()}
            print("Samples removed due to the following reasons:")
            for k,v in self.removal_reasons.items():
                print(f"{k}: {v}")
        else:
            self.sample_df = sample_df.copy()
        
        self.outcome_df = outcome_df.copy()
        self.y_date_column = y_date_column
        self.file_col = file_col
        self.forecast_window_sec = forecast_window_sec if isinstance(forecast_window_sec, list) else [forecast_window_sec]
        self.sample_df_outcome_cols = [f'outcome_val_{f}sec' for f in self.forecast_window_sec]

        self.sample_df[self.y_date_column] = self.sample_df[self.file_col].apply(lambda x: dt.datetime.strptime(zarr_record_name(x).split('-',  maxsplit=1)[1], '%Y-%m-%d-%H-%M'))
        self.sample_df['subject_id'] = self.sample_df[self.file_col].apply(lambda x: zarr_record_name(x).split('-',  maxsplit=1)[0])
        self.sample_df['unique_identifier'] = self.sample_df['subject_id'].astype(str) + '-' + self.sample_df[self.y_date_column].astype(str)

        self.sample_df.sort_values(by = ['unique_identifier'], ascending=True, inplace=True)
        self.sample_df['start_idx'] = self.sample_df['start_idx'].astype(int)
        self.sample_df['end_idx'] = self.sample_df['end_idx'].astype(int)
        self.total_samples = len(self.sample_df)

    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        # get full length x, idx can be a slice
        sample = self.sample_df.iloc[idx]
        root_grp = open_zarr_group(sample[self.file_col], mode='r')
        # forecast_max = sample['end_idx'] + self.forecast_window # indice of max window for forecast
        # forecast_max_seconds = forecast_max / self.frequency # number seconds since beginning of waveform
        
        # Get label time (already rounded to nearest minute in __init__)
        Y = sample[self.sample_df_outcome_cols].values.astype(np.int64)
        Y = torch.tensor(Y, dtype=torch.int64)
        signals = []
        for channel in self.channels:
            if channel in root_grp.array_keys():
                temp = root_grp[channel][sample['start_idx']:sample['end_idx']]
                if self.clip_interpolations is not None and channel in self.clip_interpolations:
                    temp = interpolate_nan_clip(temp, physiological_range_clip=self.clip_interpolations[channel]['phys_range'], percentile_clip=self.clip_interpolations[channel]['percentiles'])
                if self.median_filter_kernel_size is not None:
                    temp = median_filter(temp, size=self.median_filter_kernel_size, mode='nearest')
                if self.butterworth_filters is not None and channel in self.butterworth_filters:
                    freq_range = self.butterworth_filters[channel]
                    btype = 'highpass' if freq_range[0] is None else 'lowpass' if freq_range[1] is None else 'bandpass'
                    freq_range = freq_range[1] if freq_range[0] is None else freq_range[0] if freq_range[1] is None else freq_range
                    temp = butterworth(temp, freq_range=freq_range, btype=btype, fs=self.frequency, order=2)
                if self.normalize_signals:
                    temp = iqr_normalization(temp, is_spo2=False)
            else:
                temp = np.zeros((sample['end_idx'] - sample['start_idx'],))
            # if channel == 'ABP':
            #     temp = (temp < 65).astype(np.int32)
            signals.append(temp)
        X = torch.from_numpy(np.array(signals, dtype=np.float32))
        #sequence_padding_mask = torch.zeros([1, X.shape[-1]]) # channels are all the same length
        if torch.isnan(X).any():
            warnings.warn(f"X has nan values, sample_idx: {idx}")
        close_zarr_group(root_grp)
        return X,Y

In [ ]:
#| export
class SelfSupervisedDataset(Dataset):
    """
    PyTorch Dataset for self-supervised learning tasks using physiological waveform data stored in zarr format.
    This dataset class handles loading, preprocessing, and preparing samples for self-supervised learning tasks.
    """
    def __init__(self, 
                 zarr_files, # zarr files that include samples
                 channels, # channels to use
                 max_seq_len_sec=None, # maximum sequence length (in seconds) to use (this is especially relevant when you are returning both stft and raw ts data to keep them in sync)
                 sample_df=None, # dataframe indicating which indices within each zarr file includes a sample
                 sample_seq_len_sec=None, # if no sample_df, generate sequences of this length in seconds as one sample
                 sample_stride_sec=None, #  if no sample_df, seconds of overlap for samples from the same array, if seq_len_seconds == overlap_seconds, there is no overlap
                 frequency=125, # frequency of underlying data
                 butterworth_filters=None, # dictionary of low pass, high pass, and bandpass dictionary to perform on channels
                 median_filter_kernel_size=None, # size of median filter to perform on channels
                 clip_interpolations=None, # dictionary of channels:{'phys_range':..., 'percentiles':...} for filtering and interpolation of filtered values
                 constant_nan_tolerance=0.2, # tolerance for nan values in the data - 0 means no nan allowed, 1 means 100% of nans allowed
                 require_all_channels=True, # indicator to require all channels to be present in the data
                 normalize_signals=True # indicator to normalize signals to 0 mean and unit variance
                 ):
        self.max_seq_len_sec = max_seq_len_sec
        self.zarr_files = zarr_files
        self.channels = channels
        self.sample_seq_len_sec = sample_seq_len_sec
        self.frequency = frequency
        self.sample_stride_sec = sample_stride_sec
        self.clip_interpolations = clip_interpolations
        self.butterworth_filters = butterworth_filters
        self.median_filter_kernel_size = median_filter_kernel_size
        self.normalize_signals = normalize_signals
        self.constant_nan_tolerance = constant_nan_tolerance
        self.require_all_channels = require_all_channels
            
        if sample_df is None:
            assert sample_seq_len_sec is not None and sample_stride_sec is not None, "You must provide sample sequence lengths and strides if you do not pass a sample_df"
            print(f"Calculating samples with {sample_seq_len_sec} sec length and {sample_stride_sec} sec stride")
            self.sample_df, _ = calculate_samples_mp(zarr_files, channels=channels, max_seq_len_sec=max_seq_len_sec, sample_seq_len_sec=sample_seq_len_sec, frequency=frequency, stride_sec=sample_stride_sec, include_partial_samples=False, constant_nan_tolerance=constant_nan_tolerance, require_all_channels=require_all_channels)
        else:
            self.sample_df = sample_df.copy()

        self.sample_df['start_idx'] = self.sample_df['start_idx'].astype(int)
        self.sample_df['end_idx'] = self.sample_df['end_idx'].astype(int)
        self.total_samples = len(self.sample_df)

    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        # get full length x, idx can be a slice
        sample = self.sample_df.iloc[idx]
        root_grp = open_zarr_group(sample['file'], mode='r')
        
        signals = []
        for channel in self.channels:
            temp = root_grp[channel][sample['start_idx']:sample['end_idx']]
            if self.clip_interpolations is not None and channel in self.clip_interpolations:
                temp = interpolate_nan_clip(temp, physiological_range_clip=self.clip_interpolations[channel]['phys_range'], percentile_clip=self.clip_interpolations[channel]['percentiles'])
            if self.median_filter_kernel_size is not None:
                temp = median_filter(temp, size=self.median_filter_kernel_size, mode='nearest')
            if self.butterworth_filters is not None and channel in self.butterworth_filters:
                freq_range = self.butterworth_filters[channel]
                btype = 'highpass' if freq_range[0] is None else 'lowpass' if freq_range[1] is None else 'bandpass'
                freq_range = freq_range[1] if freq_range[0] is None else freq_range[0] if freq_range[1] is None else freq_range
                temp = butterworth(temp, freq_range=freq_range, btype=btype, fs=self.frequency, order=2)
            if self.normalize_signals:
                temp = iqr_normalization(temp, is_spo2=False)
            signals.append(temp)
        
        close_zarr_group(root_grp)
        X = Y = torch.from_numpy(np.array(signals, dtype=np.float32))
        if torch.isnan(X).any():
            warnings.warn(f"X has nan values, sample_idx: {idx}")
        return X,Y

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()